# Climate Atlas — projections-climate-atlas

This dataset provides climate projection indices from CMIP5 and CORDEX, covering variables such as mean temperature, precipitation, frost days, and heat extremes. Data is available at global (CMIP5) and European (CORDEX) scales under various RCP scenarios.

**Defaults:** CORDEX Europe, RCP 8.5, period 2006-2100. All request parameters are **scalar** (single strings, not arrays).

The API returns data in CoverageJSON format, which can be decoded to xarray using `covjsonkit`.

## Bounding-box query

In [ ]:
import requests

url = "https://polytope-dss.ecmwf.int/api/v1/dataset/projections-climate-atlas/extract/"
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
}
data = {
    "request": {
        "variable": "monthly_mean_of_daily_mean_temperature",
        "origin": "cordex",
        "experiment": "rcp_8_5",
        "domain": "europe",
        "period": "2006-2100",
    },
    "feature": {
        "type": "boundingbox",
        "coordinates": [[49.5, 5.5], [50.5, 6.5]],
    },
}

response = requests.post(url, headers=headers, json=data)

# Handle async responses (slow datasets return 202 with a job_id)
if response.status_code == 202:
    import time
    poll_url = "https://polytope-dss.ecmwf.int" + response.json()["poll_url"]
    while response.status_code == 202:
        time.sleep(10)
        response = requests.get(poll_url)
        print(f"Polling... status: {response.status_code}")

print(response.status_code)
print(response.json())

In [ ]:
from covjsonkit.api import Covjsonkit
decoder = Covjsonkit().decode(response.json())

ds = decoder.to_xarray()
ds

## Polygon query

In [ ]:
import requests

url = "https://polytope-dss.ecmwf.int/api/v1/dataset/projections-climate-atlas/extract/"
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
}
data = {
    "request": {
        "variable": "monthly_mean_of_daily_accumulated_precipitation",
        "origin": "cordex",
        "experiment": "rcp_8_5",
        "domain": "europe",
        "period": "2006-2100",
    },
    "feature": {
        "type": "polygon",
        "coordinates": [
            [49.5, 5.5],
            [50.5, 5.5],
            [50.5, 6.5],
            [49.5, 6.5],
            [49.5, 5.5],
        ],
    },
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 202:
    import time
    poll_url = "https://polytope-dss.ecmwf.int" + response.json()["poll_url"]
    while response.status_code == 202:
        time.sleep(10)
        response = requests.get(poll_url)
        print(f"Polling... status: {response.status_code}")

print(response.status_code)
print(response.json())

In [ ]:
from covjsonkit.api import Covjsonkit
decoder = Covjsonkit().decode(response.json())

ds = decoder.to_xarray()
ds

## Minimal request (defaults applied)

Only `variable` is required. The API defaults to CORDEX Europe, RCP 8.5, period 2006-2100.

In [ ]:
import requests

url = "https://polytope-dss.ecmwf.int/api/v1/dataset/projections-climate-atlas/extract/"
headers = {
    "accept": "application/json",
    "Content-Type": "application/json",
}
data = {
    "request": {
        "variable": "monthly_count_of_frost_days",
    },
    "feature": {
        "type": "boundingbox",
        "coordinates": [[48.0, 11.0], [48.5, 11.5]],
    },
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 202:
    import time
    poll_url = "https://polytope-dss.ecmwf.int" + response.json()["poll_url"]
    while response.status_code == 202:
        time.sleep(10)
        response = requests.get(poll_url)
        print(f"Polling... status: {response.status_code}")

print(response.status_code)
print(response.json())

In [ ]:
from covjsonkit.api import Covjsonkit
decoder = Covjsonkit().decode(response.json())

ds = decoder.to_xarray()
ds

## Available variables

Check the [constraints file](../constraints/projections-climate-atlas.json) for all valid parameter combinations.

Key variables include:
- `monthly_mean_of_daily_mean_temperature`
- `monthly_mean_of_daily_maximum_temperature`
- `monthly_mean_of_daily_minimum_temperature`
- `monthly_mean_of_daily_accumulated_precipitation`
- `monthly_count_of_frost_days`
- `annual_cooling_degree_days`
- `annual_heating_degree_days`
- `annual_consecutive_dry_days`
- `monthly_maximum_of_1_day_accumulated_precipitation`
- `monthly_maximum_of_5_day_accumulated_precipitation`

Origins: `cmip5`, `cordex`  
Experiments: `historical`, `rcp_2_6`, `rcp_4_5`, `rcp_8_5`  
Domains: `global` (CMIP5 only), `europe` (CORDEX only)